In [ ]:
!git clone https://github.com/Chetnapadhi/SentimentAnalysis.git

Cloning into 'SentimentAnalysis'...
remote: Enumerating objects: 37, done.
remote: Counting objects: 100% (37/37), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 37 (delta 0), reused 37 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (37/37), 55.52 KiB | 13.88 MiB/s, done.


In [ ]:
%cd /content/SentimentAnalysis

/content/SentimentAnalysis


In [ ]:
!ls

config.yaml  notebooks	requirements.txt  smoke_test_e0.py
data	     README.md	run_e0.py	  src


In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [ ]:
!pip install -q -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 132.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 126.7 MB/s eta 0:00:00


In [ ]:
!python -c "import torch, transformers, pandas; print('Environment OK')"

Environment OK


In [ ]:
!python -m src.data.inspect_datasets

README.md: 100% 1.39k/1.39k [00:00<00:00, 455kB/s]
train-emoji-bear-unmodified.txt: 100% 561k/561k [00:00<00:00, 114MB/s]

train-emoji-bull-unmodified.txt: downloading bytes:  65% 2.96M/4.56M [00:01<00:00, 1.96MB/s]
train-emoji-bull-unmodified.txt: downloading bytes: 100% 2.96M/2.96M [00:01<00:00, 1.96MB/s,  287kB/s  ]
train-emoji-bull-unmodified.txt: reconstructing file: 100% 4.56M/4.56M [00:01<00:00, 3.01MB/s,  441kB/s  ]

train-emoji-net-unmodified.txt: downloading bytes:   0% 0.00/2.13M [00:00<?, ?B/s]
train-emoji-net-unmodified.txt: downloading bytes: 100% 1.42M/1.42M [00:01<00:00, 990kB/s,  138kB/s  ]
train-emoji-net-unmodified.txt: reconstructing file: 100% 2.13M/2.13M [00:01<00:00, 1.48MB/s,  206kB/s  ]

train-svm-bear-neg.txt: downloading bytes:   0% 0.00/1.58M [00:00<?, ?B/s]
train-svm-bear-neg.txt: downloading bytes: 100% 1.08M/1.08M [00:01<00:00, 832kB/s,  105kB/s  ]
train-svm-bear-neg.txt: reconstructing file: 100% 1.58M/1.58M [00:01<00:00, 1.22MB/s,  154kB/s  ]

train-svm

In [ ]:
!python -m src.data.stocktwits_adapter

Creating CSV from Arrow format: 100% 211/211 [00:00<00:00, 231.42ba/s]
Saved train: 210699 rows -> data/processed/stocktwits_train.csv
Creating CSV from Arrow format: 100% 21/21 [00:00<00:00, 227.81ba/s]
Saved validation: 20676 rows -> data/processed/stocktwits_validation.csv
Creating CSV from Arrow format: 100% 12/12 [00:00<00:00, 226.67ba/s]
Saved test: 11966 rows -> data/processed/stocktwits_test.csv
Report saved -> data/inspection/stocktwits_labels.json

STOCKTWITS LABEL RECOVERY COMPLETE

TRAIN:
  HF rows:       211758
  Matched:       210699
  Dropped:       1059 (conflicting labels)
  Class dist:    {'Bearish': 35843, 'Neutral': 63593, 'Bullish': 111263}
  Emoji records: 210699

VALIDATION:
  HF rows:       20761
  Matched:       20676
  Dropped:       85 (conflicting labels)
  Class dist:    {'Bearish': 4073, 'Neutral': 7496, 'Bullish': 9107}
  Emoji records: 20676

TEST:
  HF rows:       11984
  Matched:       11966
  Dropped:       18 (conflicting labels)
  Class dist:    {'B

In [ ]:
!python -m src.data.build_final_dataset

Saved raw snapshots: stocktwits_{train,validation,test}_raw.csv

FINAL DATASET CONSTRUCTION REPORT
Training deduplication:
  HF train rows:              211758
  After conflict-label drop:   210699 (adapter output)
  Duplicate rows removed:      119551
  Train<->test overlap removed:27
  Final training rows:         91121
  Validation (unchanged):      20676
  Test (unchanged):            11966

Class distribution (train original -> final):
  Bearish: 35843 -> 7290
  Neutral: 63593 -> 26237
  Bullish: 111263 -> 57594

Class weights (train-only, final):
  Bearish: 4.1665
  Neutral: 1.1577
  Bullish: 0.5274

Manifest written -> data/processed/experiment_manifest.json


In [ ]:
!python -m src.data.preprocessing

Saved 91121 canonical rows -> data/processed/canonical/final_train.jsonl
Saved 20676 canonical rows -> data/processed/canonical/final_validation.jsonl
Saved 11966 canonical rows -> data/processed/canonical/final_test.jsonl


In [ ]:
%cd /content/SentimentAnalysis

/content/SentimentAnalysis


In [ ]:
!pwd

/content/SentimentAnalysis


In [ ]:
!sed -n '1,240p' run_e0.py

"""Google Colab launcher for the full E0 pipeline.

This orchestrates: dataset ingestion -> canonical build -> featurization ->
head training -> test evaluation -> plots -> error analysis -> research report.
Run the cells in ``colab/colab_e0_notebook.ipynb`` or execute this script after
the dataset build steps have run.

Requires a GPU runtime (T4) for reasonable featurization speed on 91K examples.
GUARDED: requires RUN_E0=1 environment variable to execute.
"""

from __future__ import annotations

import os
import sys

# GUARD: require RUN_E0=1 to prevent accidental execution
if os.environ.get("RUN_E0") != "1":
    print("GUARDED: E0 requires RUN_E0=1 environment variable.")
    print("Run as: RUN_E0=1 python run_e0.py")
    sys.exit(0)

# Ensure project root on path
ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
sys.path.insert(0, ROOT)


def main() -> None:
    # 1. Build final canonical datasets (from approved final CSVs)
    from src.data.preprocessing import b

In [ ]:
!ls data/processed/canonical

final_test.jsonl  final_train.jsonl  final_validation.jsonl


In [ ]:
!RUN_E0=1 python run_e0.py

>>> Building canonical final datasets...
Saved 91121 canonical rows -> data/processed/canonical/final_train.jsonl
Saved 20676 canonical rows -> data/processed/canonical/final_validation.jsonl
Saved 11966 canonical rows -> data/processed/canonical/final_test.jsonl
>>> Running E0 training + evaluation...
Using device: cuda
Class weights (train-only): [4.1664838790893555, 1.1576653718948364, 0.5273755192756653]
config.json: 100% 570/570 [00:00<00:00, 2.03MB/s]
tokenizer_config.json: 100% 48.0/48.0 [00:00<00:00, 275kB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 15.3MB/s]
tokenizer.json: 100% 466k/466k [00:00<00:00, 3.01MB/s]

model.safetensors: downloading bytes:  88% 389M/440M [00:02<00:00, 315MB/s, 34.4MB/s  ]
model.safetensors: reconstructing file:  76% 335M/440M [00:02<00:00, 119MB/s]
model.safetensors: downloading bytes: 100% 415M/415M [00:03<00:00, 135MB/s, 37.2MB/s  ]
model.safetensors: reconstructing file: 100% 440M/440M [00:03<00:00, 143MB/s, 40.3MB/s  ]
Loading weights: 100% 199/1

In [ ]:
!find results/E0 -maxdepth 2 -type f | sort

results/E0/best_model.pt
results/E0/config_snapshot.yaml
results/E0/confusion_matrix.png
results/E0/E0_report.md
results/E0/embeddings_cache/embeds_test_bert-base-uncased_608685ef.npy
results/E0/embeddings_cache/embeds_train_bert-base-uncased_608685ef.npy
results/E0/embeddings_cache/embeds_validation_bert-base-uncased_608685ef.npy
results/E0/error_analysis.csv
results/E0/metrics.json
results/E0/predictions.csv
results/E0/training_history.png


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!mkdir -p /content/drive/MyDrive/SentimentAnalysis/results/E0

In [ ]:
!cp -r results/E0/* /content/drive/MyDrive/SentimentAnalysis/results/E0/

In [ ]:
!find /content/drive/MyDrive/SentimentAnalysis/results/E0 -maxdepth 2 -type f | sort

/content/drive/MyDrive/SentimentAnalysis/results/E0/best_model.pt
/content/drive/MyDrive/SentimentAnalysis/results/E0/config_snapshot.yaml
/content/drive/MyDrive/SentimentAnalysis/results/E0/confusion_matrix.png
/content/drive/MyDrive/SentimentAnalysis/results/E0/E0_report.md
/content/drive/MyDrive/SentimentAnalysis/results/E0/embeddings_cache/embeds_test_bert-base-uncased_608685ef.npy
/content/drive/MyDrive/SentimentAnalysis/results/E0/embeddings_cache/embeds_train_bert-base-uncased_608685ef.npy
/content/drive/MyDrive/SentimentAnalysis/results/E0/embeddings_cache/embeds_validation_bert-base-uncased_608685ef.npy
/content/drive/MyDrive/SentimentAnalysis/results/E0/error_analysis.csv
/content/drive/MyDrive/SentimentAnalysis/results/E0/metrics.json
/content/drive/MyDrive/SentimentAnalysis/results/E0/predictions.csv
/content/drive/MyDrive/SentimentAnalysis/results/E0/training_history.png


In [ ]:
!ls data/processed/canonical

final_test.jsonl  final_train.jsonl  final_validation.jsonl


In [ ]:
!RUN_E0=1 python run_e0.py

>>> Building canonical final datasets...
Saved 91121 canonical rows -> data/processed/canonical/final_train.jsonl
Saved 20676 canonical rows -> data/processed/canonical/final_validation.jsonl
Saved 11966 canonical rows -> data/processed/canonical/final_test.jsonl
>>> Running E0 training + evaluation...
Using device: cuda
Class weights (train-only): [4.1664838790893555, 1.1576653718948364, 0.5273755192756653]
Loading weights: 100% 199/199 [00:00<00:00, 4546.68it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weig